In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra/lra-gcp/notebooks/worked")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 01 · Durable execution with the `lra` engine

Same invariants, real engine: leases, optimistic concurrency, explicit retries, a reaper. Everything runs on in-memory adapters that mimic Firestore/Cloud Tasks semantics, with a controllable clock and chaos hooks.

In [1]:
from datetime import timedelta
import json
from lra import Engine, Event, Budget, Workflow, Next, Done, Wait, StepFailed, RunStatus, StepTask, SimulatedCrash
from lra.adapters.memory import FakeClock, FakeLLM, InMemoryStateStore, InMemoryTaskQueue, InMemoryEventBus, LocalRunner
from lra.examples import ALL_WORKFLOWS
from lra.examples.scripted import research_routes

def harness(routes=None, fail_times=0, chaos=None, lease_ttl_s=60):
    """One engine over in-memory adapters, driven the way Cloud Tasks would drive it."""
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    llm = FakeLLM(routes=routes or research_routes(), fail_times=fail_times)
    engine = Engine(store=store, queue=queue, bus=bus, llm=llm, clock=clock, workflows=ALL_WORKFLOWS,
                    worker_id="worker-A", lease_ttl=timedelta(seconds=lease_ttl_s), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store, queue, bus

def show(run):
    print(f"{run.run_id} {run.status.value:12s} step={run.current_step} attempts={run.attempts} "
          f"wait={run.wait.key if run.wait else None} steps_used={run.budget.steps_used}")

## A three-step workflow

Steps return `Next`, `Done`, `Wait` or `FanOut`. They mutate `ctx.state`, call the model through `ctx.llm` (budgeted), and do side effects through `ctx.effect` (idempotent).

In [2]:
wf = Workflow("triage", version="1", default_budget=Budget(max_steps=10, max_cost_usd=0.5))

@wf.step(start=True)
def classify(ctx):
    resp = ctx.llm("Classify this ticket as billing/technical/other: " + ctx.input["ticket"])
    ctx.state["category"] = resp.text.strip()
    return Next("lookup")

@wf.step(max_attempts=3, backoff_base_s=1.0)
def lookup(ctx):
    resp = ctx.llm("Find the relevant policy for: " + ctx.state["category"])   # may 503 -> retried
    ctx.state["policy"] = resp.text
    return Next("respond")

TICKETS_UPDATED = []
@wf.step()
def respond(ctx):
    rec = ctx.effect("update-ticket", lambda: TICKETS_UPDATED.append(ctx.run_id) or {"comment_id": "c-1"})
    return Done({"category": ctx.state["category"], "comment_id": rec["comment_id"]})

routes = {"Classify": "billing", "policy": "Refunds within 14 days."}
engine, runner, clock, store, queue, bus = harness(routes=routes, fail_times=1)   # first model call fails once
engine.registry.register(wf)

## Run it one task at a time

`LocalRunner.step()` delivers one due task, exactly like one Cloud Tasks push.

In [3]:
run = engine.start("triage", {"ticket": "I was charged twice"})
show(store.get(run.run_id))
while runner.step(auto_advance=True):        # auto_advance: jump the clock to a delayed retry
    show(store.get(run.run_id))
r = store.get(run.run_id)
print("trace:", runner.trace)
print("history:", [(h.step, h.attempt, h.status) for h in r.history])
assert r.status == RunStatus.SUCCEEDED and r.attempt_of("classify") == 2 and TICKETS_UPDATED == [run.run_id]

run_9aea07c20c85 PENDING      step=classify attempts={'classify': 1} wait=None steps_used=0
run_9aea07c20c85 RUNNING      step=classify attempts={'classify': 2} wait=None steps_used=0
run_9aea07c20c85 RUNNING      step=lookup attempts={'classify': 2, 'lookup': 1} wait=None steps_used=1
run_9aea07c20c85 RUNNING      step=respond attempts={'classify': 2, 'lookup': 1, 'respond': 1} wait=None steps_used=2
run_9aea07c20c85 SUCCEEDED    step=None attempts={'classify': 2, 'lookup': 1, 'respond': 1} wait=None steps_used=3
trace: [('run_9aea07c20c85--step--classify--1', 'retry'), ('run_9aea07c20c85--step--classify--2', 'ok'), ('run_9aea07c20c85--step--lookup--1', 'ok'), ('run_9aea07c20c85--step--respond--1', 'done')]
history: [('classify', 1, 'error'), ('classify', 2, 'ok'), ('lookup', 1, 'ok'), ('respond', 1, 'ok')]


The first `classify` attempt hit the simulated 503; the engine bumped the attempt (making the old task stale), enqueued a delayed retry, and the runner fast-forwarded the clock to it. Retries live in run history, not hidden in the queue.

## Chaos: crash after the checkpoint, before the enqueue

The chaos hook raises `SimulatedCrash` at a named point. A real crash never releases its lease, so the reaper must wait for it to expire.

In [4]:
crashed = []
def chaos(point, run):
    if point == "after_commit_before_enqueue" and run.current_step == "respond" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()

engine, runner, clock, store, queue, bus = harness(routes=routes, chaos=chaos, lease_ttl_s=60)
engine.registry.register(wf)
run = engine.start("triage", {"ticket": "app crashes on login"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    print("crashed while", store.get(run.run_id).current_step, "was being enqueued")
r = store.get(run.run_id); show(r); print("lease held by:", r.lease.owner, "until", r.lease.expires_at.time())

runner.run_until_idle()                       # Cloud Tasks redelivers the crashed task -> stale
print("redelivery outcome:", runner.trace[-1][1], "| queue:", len(queue))
print("reap now:", engine.reap()["leases_recovered"])
clock.advance(seconds=61)
print("reap after lease expiry:", engine.reap()["leases_recovered"])
runner.run_until_idle()
assert store.get(run.run_id).status == RunStatus.SUCCEEDED

crashed while respond was being enqueued
run_8f5115372011 RUNNING      step=respond attempts={'classify': 1, 'lookup': 1, 'respond': 1} wait=None steps_used=2
lease held by: worker-A until 00:01:00
redelivery outcome: stale | queue: 0
reap now: []
reap after lease expiry: ['run_8f5115372011']


## Crash after the side effect, before the checkpoint

The most expensive window: the effect happened, the checkpoint didn't. On redelivery the engine re-runs the step, finds the effect record, and skips it.

In [5]:
TICKETS_UPDATED.clear(); crashed.clear()
def chaos2(point, run):
    if point == "after_step_before_commit" and run.current_step == "respond" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()
engine, runner, clock, store, queue, bus = harness(routes=routes, chaos=chaos2)
engine.registry.register(wf)
run = engine.start("triage", {"ticket": "refund please"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    pass
print("effects applied before crash:", TICKETS_UPDATED)
clock.advance(seconds=61); engine.reap(); runner.run_until_idle()
r = store.get(run.run_id)
print(r.status.value, "attempts of respond:", r.attempt_of("respond"), "effects:", TICKETS_UPDATED)
assert r.status == RunStatus.SUCCEEDED and len(TICKETS_UPDATED) == 1

effects applied before crash: ['run_daa40bb11075']
SUCCEEDED attempts of respond: 1 effects: ['run_daa40bb11075']


## Leases: a second worker

Two replicas receive the same task 50 ms apart. The loser gets `lease-held`, which the HTTP layer maps to **503** so Cloud Tasks retries later.

In [6]:
engine, runner, clock, store, queue, bus = harness(routes=routes)
engine.registry.register(wf)
other = Engine(store=store, queue=queue, bus=bus, llm=engine.llm, clock=clock, workflows=engine.registry, worker_id="worker-B")
run = engine.start("triage", {"ticket": "x"})
task = queue.pop_due()
store.acquire_lease(run.run_id, "worker-A", timedelta(seconds=60), clock.now())      # A is mid-step
print("B tries:", other.execute_task(task))
clock.advance(seconds=61)                                                             # A died
print("B after expiry:", other.execute_task(task))
assert store.get(run.run_id).current_step == "lookup"

B tries: lease-held
B after expiry: ok


## Takeaways

* Retries are **explicit attempts**; the old task becomes stale by construction.
* Two crash windows, two recovery mechanisms: reaper (lost enqueue) and effect records (lost checkpoint).
* The worker is stateless — every replica can execute any step; the lease is the only coordination.